# 9. 객체 탐지(Object Detection)와 YOLO

> 이 노트북은 강의 자료를 기반으로 작성되었습니다.  
> Google Colab에서 `런타임 > 런타임 유형 변경 > T4 GPU`로 설정 후 실행하세요.

## 학습 목표

1. 객체 탐지의 개념과 Bounding Box, IOU를 설명할 수 있다
2. NMS의 동작 과정과 필요성을 이해할 수 있다
3. 정밀도, 재현율, AP, mAP 지표를 해석할 수 있다
4. 객체 탐지 모델의 발전 흐름(RCNN → YOLO)을 설명할 수 있다
5. Ultralytics YOLO를 사용하여 커스텀 데이터셋으로 모델을 학습할 수 있다
6. 웹캠을 활용한 실시간 객체 탐지를 구현할 수 있다

## 진행 순서

1. [객체 탐지 개요](#part1)
2. [NMS와 모델 성능 평가](#part2)
3. [객체 탐지의 발전](#part3)
4. [YOLO 실습: Ultralytics](#part4)
5. [YOLO 실습: 웹캠 객체 탐지](#part5)
6. [더 알아보기: YOLO 너머의 객체 탐지](#part6)
7. [통합 정리](#part7)

---

<a id="part1"></a>
## 1. 객체 탐지 개요

**학습목표**: 객체 탐지의 개념과 Bounding Box, IOU를 설명할 수 있다

- 한 이미지에서 객체와 그 경계 상자(bounding box)를 탐지
- 객체 탐지 알고리즘은 일반적으로 이미지를 입력으로 받고, 경계 상자와 객체 클래스 리스트를 출력
- 경계 상자에 대해 그에 대응하는 예측 클래스와 클래스의 신뢰도(confidence)를 출력

### 활용분야

- 자율 주행 자동차에서 다른 자동차와 보행자를 찾을 때
- 의료 분야에서 방사선 사진을 사용해 종양이나 위험한 조직을 찾을 때
- 제조업에서 조립 로봇이 제품을 조립하거나 수리할 때
- 보안 산업에서 위협을 탐지하거나 사람 수를 셀 때

### Bounding Box

- 이미지에서 하나의 객체 전체를 포함하는 가장 작은 직사각형

![](./img/yolo/object_detaction001.png)

### IOU(Intersection Over Union)

- 실측값(Ground Truth)과 모델이 예측한 값이 얼마나 겹치는지를 나타내는 지표

  ![](./img/yolo/object_detaction002.png)

- IOU가 높을수록 잘 예측한 모델

  ![](./img/yolo/object_detaction003.png)

- 예시

  ![](./img/yolo/object_detaction004.png)

---

**핵심**: 객체 탐지는 이미지에서 객체의 위치(Bounding Box)와 클래스를 동시에 예측한다. IOU는 예측과 실측의 겹침 비율로, 탐지 품질을 측정하는 핵심 지표이다.

> **확인**: IOU가 0.3이면 좋은 탐지라고 할 수 있을까요? 일반적으로 IOU 0.5 이상을 "정답"으로 간주합니다.

---

<a id="part2"></a>
## 2. NMS와 모델 성능 평가

**학습목표**: NMS의 동작 과정과 필요성을 이해하고, 정밀도, 재현율, AP, mAP 지표를 해석할 수 있다

### NMS(Non-Maximum Suppression, 비최댓값 억제)

- 확률이 가장 높은 상자와 겹치는 상자들을 제거하는 과정
- 최댓값을 갖지 않는 상자들을 제거
- 과정
  1. 확률 기준으로 모든 상자를 정렬하고 먼저 가장 확률이 높은 상자를 취함
  2. 각 상자에 대해 다른 모든 상자와의 IOU를 계산
  3. 특정 임곗값을 넘는 상자는 제거

  ![](./img/yolo/object_detaction005.png)

### 모델 성능 평가

#### 정밀도(Precision)와 재현율(Recall)

AP, mAP 등 객체 탐지 평가 지표의 기본이 되는 개념입니다.

먼저 예측 결과를 4가지로 분류합니다.

|  | 실제 객체 있음 | 실제 객체 없음 |
|---|:---:|:---:|
| **모델이 "있다" 예측** | TP (정탐) | FP (오탐) |
| **모델이 "없다" 예측** | FN (미탐) | TN (정상 무시) |

- **True Positive (TP)**: 객체가 있는데 있다고 예측 → 정답
- **False Positive (FP)**: 객체가 없는데 있다고 예측 → 오탐 (허위 경보)
- **False Negative (FN)**: 객체가 있는데 놓침 → 미탐
- **True Negative (TN)**: 객체가 없는데 없다고 예측 → 정답 (객체 탐지에서는 배경 영역이 대부분이라 TN은 매우 많아서 평가 지표에는 잘 쓰지 않음)

이를 바탕으로 정밀도와 재현율을 계산합니다.

```
                  TP                              TP
Precision = ───────────           Recall = ───────────
              TP + FP                        TP + FN
```

- **Precision(정밀도)** = "모델이 탐지한 것 중 실제로 맞은 비율" — FP가 많으면 정밀도 ↓
- **Recall(재현율)** = "실제 객체 중 모델이 찾아낸 비율" — FN이 많으면 재현율 ↓

> **직관적 비유**
> - 보행자 탐지 → **재현율**이 중요: 보행자를 놓치면(FN) 사고 위험. 차가 가끔 멈추더라도(FP) 안전이 우선
> - 투자 기회 탐지 → **정밀도**가 중요: 잘못된 기회에 투자하면(FP) 손실. 일부 기회를 놓치더라도(FN) 신중해야 함

#### 정밀도-재현율 곡선(Precision-Recall Curve)

- 신뢰도 임계값마다 모델의 정밀도와 재현율을 시각화
- 모든 bounding box와 함께 모델이 예측의 정확성을 얼마나 확실하는지 0 ~ 1사이의 숫자로 나타내는 신뢰도를 출력
- 임계값 T에 따라 정밀도와 재현율이 달라짐
  - 임계값 T 이하의 예측은 제거함
  - T가 1에 가까우면 정밀도는 높지만 재현율은 낮음 — 신뢰도가 높은 예측만 유지하기 때문
  - T가 0에 가까우면 정밀도는 낮지만 재현율은 높음 — 대부분의 예측을 유지하기 때문

![](./img/yolo/object_detaction006.png)

#### AP (Average Precision, 평균 정밀도) 와 mAP(mean Average Precision)

- 곡선의 아래 영역에 해당
- 항상 1x1 정사각형으로 구성되어 있음 — 즉, 항상 0 ~ 1 사이의 값을 가짐
- 단일 클래스에 대한 모델 성능 정보를 제공
- 전역 점수를 얻기 위해서 mAP를 사용
- 예를 들어, 데이터셋이 10개의 클래스로 구성된다면 각 클래스에 대한 AP를 계산하고, 그 숫자들의 평균을 다시 구함

#### VOC와 COCO의 mAP 차이

대표적인 두 벤치마크 데이터셋은 평가 방식이 다릅니다.

| 벤치마크 | 클래스 수 | IOU 임계값 | 평가 방식 |
|----------|----------|-----------|----------|
| **VOC** | 20개 | 0.5 (단일) | mAP@0.5 — IOU 50%만 넘으면 정답 |
| **COCO** | 80개 | 0.5 ~ 0.95 (10단계) | mAP@[.5:.95] — 10개 임계값의 평균 |

COCO의 mAP 점수가 VOC보다 보통 낮게 나오는 **핵심 원인은 IOU 임계값**입니다. COCO는 IOU 0.5부터 0.95까지 0.05 간격으로 10개 임계값에서 각각 AP를 계산한 후 평균을 냅니다. 즉, 박스가 거의 정확히 맞아야(IOU 0.9 이상) 높은 점수를 받을 수 있어서 더 엄격합니다.

![](./img/yolo/object_detaction007.png)

---

**핵심**: NMS는 중복된 예측 상자를 제거하여 최종 탐지 결과를 정리한다. mAP는 모든 클래스의 AP 평균으로, 객체 탐지 모델의 종합 성능을 나타낸다.

> **확인**: COCO에서 mAP 50이면 VOC에서는 보통 더 높은 점수가 나옵니다. 왜 그럴까요?

---

<a id="part3"></a>
## 3. 객체 탐지의 발전

**학습목표**: 객체 탐지 모델의 발전 흐름(RCNN → YOLO)을 설명할 수 있다

![](./img/yolo/object_detaction010.png)

객체 탐지 모델은 크게 **2-stage**(영역 제안 후 분류)와 **1-stage**(한 번에 탐지) 방식으로 나뉩니다.

### 2-Stage Detector: "정확하지만 느림"

| 모델 | 연도 | 핵심 아이디어 | 한계 |
|------|------|-------------|------|
| **RCNN** | 2013 | 영역 후보(Region Proposal)를 CNN으로 분류 | 후보마다 CNN 통과 → 매우 느림 |
| SPP Net | 2014 | CNN 1회 실행 후 영역별 풀링 | 학습 과정이 복잡 |
| Fast RCNN | 2015 | ROI Pooling으로 단순화, 분류+위치 동시 학습 | 영역 제안이 여전히 별도 |
| **Faster RCNN** | 2015 | RPN으로 영역 제안까지 네트워크에 통합 (end-to-end) | 실시간에는 부족 (~7 fps) |

> **핵심 흐름**: RCNN(느림) → Fast RCNN(빨라짐) → Faster RCNN(end-to-end) 순서로 **영역 제안 단계를 점점 네트워크 안으로 통합**하며 속도를 개선했습니다.

### 1-Stage Detector: "빠르고 실시간"

| 모델 | 연도 | 핵심 아이디어 |
|------|------|-------------|
| **YOLOv1** | 2015 | 이미지를 그리드로 나눠 한 번에 탐지. 최초의 1-stage detector |
| YOLOv2 | 2016 | Batch Norm, Anchor Box 도입으로 정확도 개선 |
| **SSD** | 2016 | 다중 스케일 Feature Map으로 작은 객체도 탐지. 30~40 fps 달성 |
| YOLOv3 | 2018 | FPN 도입으로 다중 스케일 탐지, 정확도 대폭 향상 |
| RetinaNet | 2017 | Focal Loss로 배경/객체 불균형 해결. 1-stage이면서 2-stage급 정확도 달성 |

> **핵심 흐름**: 1-stage 방식은 **영역 제안 없이 이미지 전체를 한 번에 처리**하여 실시간 탐지가 가능해졌습니다.

### YOLO의 진화

| 버전 | 연도 | 발표자 | 주요 변화 |
|------|------|--------|----------|
| YOLOv1 | 2015 | Redmon et al. | 최초 1-stage detector |
| YOLOv2 | 2016 | Redmon & Farhadi | Anchor Box, Batch Norm |
| YOLOv3 | 2018 | Redmon & Farhadi | FPN, 다중 스케일 |
| YOLOv4 | 2020 | Bochkovskiy et al. | CSPDarknet53, 다양한 기법 조합으로 AP/FPS 각 10%+↑ |
| YOLOv5 | 2020 | Glenn Jocher (Ultralytics) | PyTorch 구현, 경량화 |
| YOLOv6 | 2022 | **Meituan(메이투안)** | **EfficientRep** 백본, Anchor-Free 구조 |
| YOLOv7 | 2022 | Wang, Bochkovskiy, Liao | E-ELAN, 속도-정확도 균형 SOTA |
| YOLOv8 | 2023 | Ultralytics | 모듈화, 탐지/분할/분류 통합 프레임워크 |
| YOLOv9 | 2024 | Wang & Liao | PGI, GELAN 도입 |
| YOLOv10 | 2024 | 칭화대 | NMS-free end-to-end |
| **YOLO11** | **2024.09** | **Ultralytics** | "v" 접두사 삭제. **이 실습에서 사용** |
| YOLO12 | 2025 | 커뮤니티 | Attention 기반 아키텍처. 연구/벤치마크용 |
| **YOLO26** | **2026.01** | **Ultralytics** | NMS-free end-to-end, 엣지 최적화. **최신 공식 모델** |

> **참고**: YOLOv5 이후부터는 원저자(Redmon)가 아닌 다양한 팀이 독립적으로 발전시키고 있습니다. Ultralytics는 YOLOv5, v8, YOLO11, YOLO26을 개발했고, 이 실습에서 사용하는 프레임워크입니다.

---

**핵심**: 객체 탐지는 2-stage(RCNN 계열)에서 1-stage(SSD, YOLO)로 발전하며 속도와 정확도의 균형을 개선해 왔다. 현재 YOLO 시리즈는 여러 팀에 의해 빠르게 발전하고 있다.

> **확인**: 2-stage와 1-stage의 가장 큰 차이는 무엇인가요? 어떤 상황에서 어떤 방식이 유리할까요?

---

<a id="part4"></a>
## 4. YOLO 실습: Ultralytics

**학습목표**: Ultralytics YOLO를 사용하여 커스텀 데이터셋으로 모델을 학습할 수 있다

### Ultralytics 소개

Ultralytics는 YOLOv5부터 YOLO 모델의 개발과 지원을 시작했습니다. 객체 탐지, 분류, 분할 등 다양한 작업을 하나의 프레임워크에서 수행할 수 있습니다.

이 실습에서는 **YOLO11 nano 모델(`yolo11n.pt`)**을 사용합니다. nano는 가장 가벼운 모델로, 무료 Colab 환경에서도 빠르게 실행됩니다.

Ultralytics를 설치합니다.

In [ ]:
!pip install ultralytics

### 사전학습 모델로 예측 체험

먼저 이미 학습된 COCO 모델로 이미지 탐지를 체험합니다.

In [ ]:
!yolo detect predict model=yolo11n.pt source='https://ultralytics.com/images/bus.jpg'

탐지 결과 이미지를 확인합니다.

In [ ]:
from IPython.display import Image
Image(filename='runs/detect/predict/bus.jpg', width=600)

### 포트홀 탐지 모델 - 커스텀 학습

이제 **포트홀(도로 파임) 데이터셋**으로 YOLO를 직접 학습해 봅니다.

데이터셋 출처: [Roboflow Pothole Dataset](https://public.roboflow.com/object-detection/pothole)

데이터셋을 다운로드하고 구조를 확인합니다.

In [ ]:
%mkdir -p /content/datasets/pothole
%cd /content/datasets/pothole
!curl -L "https://public.roboflow.com/ds/z4Ay3aTtU2?key=c1LsIm90PW" > roboflow.zip; unzip -q roboflow.zip; rm roboflow.zip

train / val / test 이미지 수를 확인합니다.

In [ ]:
from glob import glob

train_img_list = glob('/content/datasets/pothole/train/images/*.jpg')
test_img_list = glob('/content/datasets/pothole/test/images/*.jpg')
valid_img_list = glob('/content/datasets/pothole/valid/images/*.jpg')
print('train:', len(train_img_list), 'valid:', len(valid_img_list), 'test:', len(test_img_list))

#### data.yaml 구조 확인

YOLO 커스텀 학습의 핵심 설정 파일입니다. `data.yaml`에는 다음 정보가 들어 있습니다.

| 필드 | 역할 | 예시 |
|------|------|------|
| `train` | 학습 이미지 경로 | `train/images` |
| `val` | 검증 이미지 경로 | `valid/images` |
| `test` | 테스트 이미지 경로 (선택) | `test/images` |
| `nc` | 클래스 수 | `1` |
| `names` | 클래스 이름 목록 | `['pothole']` |

In [ ]:
%cat /content/datasets/pothole/data.yaml

#### 학습 실행

100 에폭, 이미지 크기 640으로 학습합니다. Colab T4 GPU 기준 약 30~40분 소요됩니다.

In [ ]:
%cd /content/
!yolo detect train data=/content/datasets/pothole/data.yaml model=yolo11n.pt epochs=100 imgsz=640

학습 결과를 확인합니다. `runs/detect/train/` 폴더에 학습 곡선, 예측 샘플 등이 저장됩니다.

In [ ]:
from IPython.display import Image

# 학습 중 배치 시각화
Image(filename='runs/detect/train/train_batch2702.jpg', width=1000)

In [ ]:
# 검증 배치 라벨 확인
Image(filename='runs/detect/train/val_batch2_labels.jpg', width=1000)

#### 검증 및 모델 저장

학습된 best 모델로 검증 세트를 평가합니다.

In [ ]:
!yolo detect val data=/content/datasets/pothole/data.yaml model=runs/detect/train/weights/best.pt

In [ ]:
Image(filename='runs/detect/val/val_batch2_pred.jpg', width=1000)

학습된 모델을 Google Drive에 저장하여 런타임이 끊겨도 보존합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%mkdir -p '/content/drive/MyDrive/Colab Notebooks/pytorch_test/pothole'
%cp runs/detect/train/weights/best.pt '/content/drive/MyDrive/Colab Notebooks/pytorch_test/pothole'

---

**핵심**: Ultralytics YOLO는 간단한 CLI 명령으로 학습, 검증, 예측을 수행할 수 있다. 커스텀 데이터셋(포트홀 등)으로 Fine-tuning이 가능하다.

> **확인**: `data.yaml`에서 `nc`와 `names`를 바꾸면 어떤 데이터셋이든 학습할 수 있을까요?

---

<a id="part5"></a>
## 5. YOLO 실습: 웹캠 객체 탐지

**학습목표**: 웹캠을 활용한 실시간 객체 탐지를 구현할 수 있다

### 방법 1: Colab에서 웹캠 캡처 (Colab 전용)

Colab은 웹캠에 직접 접근할 수 없으므로, JavaScript로 브라우저 웹캠을 캡처한 후 Python으로 전달합니다.

In [ ]:
from ultralytics import YOLO
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
import cv2
import numpy as np
from PIL import Image

def capture_image():
    """브라우저 웹캠에서 이미지 1장을 캡처합니다."""
    js = Javascript('''
    async function captureImage() {
        const video = document.createElement('video');
        const stream = await navigator.mediaDevices.getUserMedia({ video: true });
        video.srcObject = stream;
        await new Promise((resolve) => { video.onloadedmetadata = () => resolve(); });
        video.play();
        await new Promise((resolve) => setTimeout(resolve, 1000));
        const canvas = document.createElement('canvas');
        canvas.width = video.videoWidth;
        canvas.height = video.videoHeight;
        canvas.getContext('2d').drawImage(video, 0, 0, canvas.width, canvas.height);
        stream.getVideoTracks()[0].stop();
        return canvas.toDataURL('image/jpeg', 0.8);
    }
    captureImage();
    ''')
    display(js)
    data = eval_js('captureImage()')
    return data

def decode_image(data):
    """base64 데이터를 OpenCV 이미지로 변환합니다."""
    img_str = b64decode(data.split(',')[1])
    np_arr = np.frombuffer(img_str, np.uint8)
    return cv2.imdecode(np_arr, cv2.IMREAD_COLOR)

# YOLO 모델 로드
model = YOLO("yolo11n.pt")

# 웹캠에서 이미지 캡처 → 탐지 → 시각화
data = capture_image()
img = decode_image(data)
results = model(img)

result = results[0]
if result.boxes:
    print("탐지된 객체가 있습니다.")
    boxes = result.boxes.xyxy.cpu().numpy()
    confs = result.boxes.conf.cpu().numpy()
    class_ids = result.boxes.cls.cpu().numpy().astype(int)

    # model.names로 클래스 이름을 자동으로 가져옵니다
    names = result.names

    for box, conf, class_id in zip(boxes, confs, class_ids):
        x_min, y_min, x_max, y_max = map(int, box)
        label = f"{names[class_id]} {conf:.2f}"
        cv2.rectangle(img, (x_min, y_min), (x_max, y_max), (0, 255, 0), 2)
        cv2.putText(img, label, (x_min, y_min - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

    display(Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)))
else:
    print("탐지된 객체가 없습니다.")

### 방법 2: 로컬 PC에서 실시간 웹캠 탐지 (로컬 전용)

로컬 환경에서는 OpenCV로 웹캠에 직접 접근할 수 있습니다. `q` 키를 누르면 종료됩니다.

> **사전 설치**: `pip install ultralytics supervision`

In [ ]:
import cv2
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

def process_webcam():
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("Error: Could not open webcam.")
        return

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        results = model(frame)[0]

        # model.names를 사용하여 클래스 이름을 자동으로 가져옵니다
        for box, conf, cls_id in zip(
            results.boxes.xyxy.cpu().numpy(),
            results.boxes.conf.cpu().numpy(),
            results.boxes.cls.cpu().numpy().astype(int)
        ):
            label = f"{model.names[cls_id]}: {conf:.2f}"
            x1, y1, x2, y2 = map(int, box)
            cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 0, 0), 2)
            cv2.putText(frame, label, (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 0, 0), 2)

        cv2.imshow("YOLO Webcam", frame)
        if cv2.waitKey(25) & 0xFF == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    process_webcam()

---

**핵심**: YOLO 모델을 웹캠과 연결하면 실시간 객체 탐지를 구현할 수 있다. Colab에서는 JavaScript 캡처 방식, 로컬에서는 OpenCV 방식을 사용한다.

---

<a id="part6"></a>
## 6. 더 알아보기: YOLO 너머의 객체 탐지

지금까지 YOLO로 "학습된 클래스만" 탐지했습니다. 그런데 **학습한 적 없는 객체**도 탐지할 수 있다면 어떨까요?

### YOLO-World: 텍스트로 탐지 대상을 지정

YOLO-World는 텍스트 프롬프트만으로 원하는 객체를 탐지합니다. 포트홀 데이터셋으로 100 에폭 학습할 필요 없이, 텍스트 한 줄로 바로 탐지할 수 있습니다.

In [ ]:
from ultralytics import YOLOWorld

model = YOLOWorld("yolov8s-world.pt")

# 탐지하고 싶은 클래스를 텍스트로 지정 — 학습 없이!
model.set_classes(["pothole", "crack", "manhole cover"])

results = model("test_road.jpg")
results[0].show()

### 객체 탐지 모델의 현재 지형도

| 모델 | 방식 | 특징 | 언제 쓰나? |
|------|------|------|-----------|
| **YOLO11** | CNN 기반 | 빠르고 가벼움. 실시간 탐지에 최적 | 속도가 중요할 때 |
| **RT-DETR** | Transformer 기반 | NMS 불필요, end-to-end | 정확도 + 실시간 둘 다 필요할 때 |
| **RF-DETR** | Transformer 기반 | COCO 60.5 mAP. 실시간 모델 중 최고 정확도 | 최고 정확도가 필요할 때 |
| **YOLO-World** | CNN + 텍스트 | 학습 없이 텍스트로 새 클래스 탐지 | 새로운 클래스를 빠르게 시도할 때 |
| **SAM 2** | Transformer 기반 | 클릭/박스로 어떤 객체든 분할 | 정밀한 윤곽선(마스크)이 필요할 때 |
| **Grounding DINO** | Transformer + 텍스트 | "소화기를 찾아줘" 같은 자연어로 탐지 | 오픈 어휘 탐지 연구 |

> **흐름 정리**: CNN 기반 YOLO(속도) → Transformer 기반 DETR(정확도) → 텍스트 기반 Open-Vocabulary(유연성) 순으로 발전하고 있습니다.

---

<a id="part7"></a>
## 7. 통합 정리

### 복습 질문

1. Bounding Box와 IOU는 각각 무엇을 의미하는가?
2. NMS가 없으면 탐지 결과가 어떻게 달라지는가?
3. 1-stage detector와 2-stage detector의 차이는 무엇인가?
4. YOLO가 실시간 탐지에 적합한 이유는 무엇인가?
5. mAP 점수가 높다고 항상 좋은 모델인가?
6. YOLO-World처럼 학습 없이 탐지하는 모델의 장단점은 무엇일까?

### 심화 과제

- confidence threshold를 0.25, 0.5, 0.75로 바꿔서 탐지 결과가 어떻게 달라지는지 비교해 보기
- YOLO11의 모델 크기(`yolo11n`, `yolo11s`, `yolo11m`)를 바꿔서 속도와 정확도 차이를 비교해 보기
- 최신 YOLO26 모델(`yolo26n.pt`)로 교체하여 YOLO11과 성능을 비교해 보기
- YOLO-World로 자유롭게 클래스를 바꿔가며 탐지해 보기
- 다른 데이터셋(COCO 부분집합, 자체 데이터)으로 YOLO 학습해 보기
- 탐지 결과를 영상 파일로 저장해 보기

### 참고 자료

- [Ultralytics YOLO 공식 문서](https://docs.ultralytics.com/)
- [Ultralytics YOLO11 모델 가이드](https://docs.ultralytics.com/models/yolo11/)
- [Ultralytics YOLO26 모델 가이드](https://docs.ultralytics.com/models/yolo26/)
- [COCO Dataset](https://cocodataset.org/)
- [Roboflow - 커스텀 데이터셋](https://roboflow.com/)
- [Pascal VOC Dataset](http://host.robots.ox.ac.uk/pascal/VOC/)